In [1]:
from pathlib import Path

import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

/home/administrateur/projects/ESTP GenAI Group 2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------------------------------------------------
# 1. Load documents
# ---------------------------------------------------------

DOCUMENTS_DIR = Path("data")

documents = []

for path in DOCUMENTS_DIR.glob("*.md"):
    text = path.read_text(encoding="utf-8")

    documents.append({
        "source": path.name,
        "text": text,
    })

print(f"Loaded {len(documents)} documents")


Loaded 15 documents


In [3]:
# ---------------------------------------------------------
# 2. Chunk documents
# ---------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


chunks = []

for document in documents:
    document_chunks = chunk_text(document["text"])

    for i, chunk in enumerate(document_chunks):
        chunks.append({
            "source": document["source"],
            "chunk_id": i,
            "text": chunk,
        })

print(f"Created {len(chunks)} chunks")

Created 82 chunks


In [4]:
# ---------------------------------------------------------
# 3. Create embeddings
# ---------------------------------------------------------

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

print(f"Embedding shape: {embeddings.shape}")



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9536.21it/s]


Embedding shape: (82, 384)


In [5]:
# ---------------------------------------------------------
# 4. Create FAISS index
# ---------------------------------------------------------

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print(f"FAISS contains {index.ntotal} vectors")


FAISS contains 82 vectors


In [6]:
# ---------------------------------------------------------
# 5. Retrieve relevant chunks
# ---------------------------------------------------------

def retrieve(question, k=3):

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, index_number in zip(scores[0], indices[0]):
        chunk = chunks[index_number]

        results.append({
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "score": float(score),
        })

    return results


In [7]:
# ---------------------------------------------------------
# 6. Connect to LM Studio
# ---------------------------------------------------------

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"
)

In [8]:
# ---------------------------------------------------------
# 7. Ask Qwen using retrieved context
# ---------------------------------------------------------

def answer_question(question):

    results = retrieve(question, k=3)

    context = "\n\n".join(
        f"[Source: {r['source']}]\n{r['text']}"
        for r in results
    )

    prompt = f"""
Answer the user's question using ONLY the information
contained in the context below.

If the answer cannot be found in the context, say:
"I don't know based on the provided documents."

Do not invent information.

Context:
----------------
{context}
----------------

Question:
{question}
"""

    response = client.chat.completions.create(
        model="mistral-7b",
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
    )

    answer = response.choices[0].message.content

    return answer, results

In [11]:
# ---------------------------------------------------------
# 8. Evaluate one question
# ---------------------------------------------------------

import json

EVALUATION_FILE = Path("evaluation/travel_questions.jsonl")
QUESTION_ID = "fact-03"

with EVALUATION_FILE.open(encoding="utf-8") as evaluation_file:
    evaluation_cases = [
        json.loads(line)
        for line in evaluation_file
        if line.strip()
    ]

evaluation_case = next(
    (case for case in evaluation_cases if case["id"] == QUESTION_ID),
    None,
)

if evaluation_case is None:
    available_ids = ", ".join(case["id"] for case in evaluation_cases)
    raise ValueError(
        f"Unknown QUESTION_ID {QUESTION_ID!r}. "
        f"Available IDs: {available_ids}"
    )

answer, results = answer_question(evaluation_case["question"])

print("--- Evaluation case ---")
print(f"ID: {evaluation_case['id']}")
print(f"Category: {evaluation_case['category']}")
print(f"Question: {evaluation_case['question']}")

print("\n--- Generated answer ---")
print(answer)

print("\n--- Expected answer points ---")
for point in evaluation_case["expected_answer_points"]:
    print(f"- {point}")

print("\n--- Expected sources ---")
for source in evaluation_case["expected_sources"]:
    print(f"- {source}")

print("\n--- Retrieved chunks ---")
for result in results:
    print(
        f"\n{result['source']} "
        f"(chunk={result['chunk_id']}, score={result['score']:.3f})"
    )
    print(result["text"])

--- Evaluation case ---
ID: fact-03
Category: factual
Question: Under what conditions may first class be booked for rail travel?

--- Generated answer ---
 First class may be booked for rail travel when one of the following conditions is met:
1. The single scheduled rail journey lasts more than four hours.
2. The employee needs to work confidentially.
3. First class costs no more than EUR 40 above the available flexible standard fare.
However, manager approval is required before purchasing first class.

--- Expected answer points ---
- The journey is longer than four hours and confidential work is needed, or first class is no more than EUR 40 above the flexible standard fare.
- Manager approval is required.

--- Expected sources ---
- rail_travel.md

--- Retrieved chunks ---

rail_travel.md (chunk=0, score=0.823)
# Rail Travel

Rail should be considered when it is a practical alternative to flying or driving. Employees should compare terminal access, transfers, and productive travel ti